<a href="https://www.kaggle.com/code/shravankumarpandey/x-post-generator-using-langgraph?scriptVersionId=335691381" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# X Post Generator using LangGraph

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GROQ_API_KEY")

In [ ]:
pip install langchain-groq

# 1.Importing Libraries

In [ ]:
from langgraph.graph import StateGraph,START,END
from langchain_core.messages import HumanMessage,SystemMessage
from typing import TypedDict,Literal,Annotated
from langchain_groq import ChatGroq
from pydantic import Field,BaseModel
import operator

# 2. Defining Model

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=secret_value_0
)

# 3. Defining Tweet Evalution Schema

In [ ]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

# 3.1 Getting Structured Output

In [ ]:
model=llm.with_structured_output(TweetEvaluation)

# 4.Defining State

In [ ]:
class TweetState(TypedDict):
    topic:str
    tweet:str
    evaluation:str
    feedack:str
    iteration:int
    max_iteration:int

# 5. Defining Node

## 5.1 Defining Generate Tweet node

In [ ]:
def generate_tweet(state:TweetState)->TweetState:
    # prompt
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]
    
    tweet=llm.invoke(messages).content
    return {"tweet":tweet}

## 5.2 Defining EvaluateTweet Node

In [ ]:
def evaluate_tweet(state:TweetState)->TweetState:
    #prompt
    # prompt
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]

    response=model.invoke(messages)
    return {"evaluation":response.evaluation,"feedback":response.feedback}
    


## 5.3 Defining Optimize Tweet Node

In [ ]:
def optimize_tweet(state:TweetState)->TweetState:
    # prompt
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]
    response=llm.invoke(messages).content
    iteration = state['iteration'] + 1
    return {"tweet":response,"iteration":iteration}
    
    

## 5.4 Route Evalution

In [ ]:
def route_evaluation(state: TweetState):

    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

# 6. Defining Graph

In [ ]:
graph=StateGraph(TweetState)

# 7.Adding Node to Graph

In [ ]:
graph.add_node("generate",generate_tweet)
graph.add_node("evaluate",evaluate_tweet)
graph.add_node("optimize",optimize_tweet)

# 8.Adding Edges to the Graph

In [ ]:
graph.add_edge(START,"generate")
graph.add_edge("generate","evaluate")
graph.add_conditional_edges('evaluate', route_evaluation, {'approved': END, 'needs_improvement': 'optimize'})
graph.add_edge("optimize","evaluate")

# 9.Compiling the Graph

In [ ]:
workflow=graph.compile()

In [ ]:
workflow

# 10.Initial State

In [ ]:
initial_state={
    "topic":"E20 in India",
    "iteration":1,
    "max_iteration":5
}

# 11.Result

In [ ]:
result=workflow.invoke(initial_state)

In [ ]:
print(result)